In [3]:
import sys
!{sys.executable} -m pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [5]:
import pandas as pd

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────
df = pd.read_csv("catalogue_raw.csv")

# ── 2. DUPLICATE DETECTION ────────────────────────────────────────────────────
df["Is_Duplicate"] = df.duplicated(subset="SKU_ID", keep="first")

# ── 3. SHORT DESCRIPTION FLAG ─────────────────────────────────────────────────
df["Short_Description"] = df["Description"].apply(
    lambda x: True if isinstance(x, str) and 0 < len(x.strip()) < 50 else False
)

# ── 4. COMPLETENESS SCORE ─────────────────────────────────────────────────────
scored_fields = ["SKU_ID", "Product_Name", "Category", "Brand",
                 "Description", "Price", "Image_URL", "Stock_Status", "Keywords"]

def completeness_score(row):
    filled = sum(
        1 for field in scored_fields
        if pd.notna(row[field]) and str(row[field]).strip() != ""
    )
    return round((filled / len(scored_fields)) * 100, 1)

df["Completeness_Score"] = df.apply(completeness_score, axis=1)

# ── 5. QUALITY FLAG ───────────────────────────────────────────────────────────
def quality_flag(row):
    if row["Is_Duplicate"]:
        return "Duplicate"
    elif row["Completeness_Score"] == 100 and not row["Short_Description"]:
        return "Good"
    elif row["Completeness_Score"] >= 70 and not row["Short_Description"]:
        return "Needs Review"
    else:
        return "Poor"

df["Quality_Flag"] = df.apply(quality_flag, axis=1)

# ── 6. SUMMARY STATS ──────────────────────────────────────────────────────────
total_skus        = len(df)
unique_skus       = df["SKU_ID"].nunique()
duplicates        = int(df["Is_Duplicate"].sum())
good              = int((df["Quality_Flag"] == "Good").sum())
needs_review      = int((df["Quality_Flag"] == "Needs Review").sum())
poor              = int((df["Quality_Flag"] == "Poor").sum())
avg_score         = round(df["Completeness_Score"].mean(), 1)

missing_per_field = {
    field: int(df[field].isna().sum() + (df[field] == "").sum())
    for field in scored_fields
}

# ── 7. BUILD EXCEL REPORT ─────────────────────────────────────────────────────
wb = Workbook()

# ── COLOR PALETTE ─────────────────────────────────────────────────────────────
GREEN  = PatternFill("solid", fgColor="C6EFCE")
YELLOW = PatternFill("solid", fgColor="FFEB9C")
RED    = PatternFill("solid", fgColor="FFC7CE")
BLUE   = PatternFill("solid", fgColor="BDD7EE")
HEADER = PatternFill("solid", fgColor="1F4E79")
GREY   = PatternFill("solid", fgColor="F2F2F2")

header_font    = Font(bold=True, color="FFFFFF", size=11)
title_font     = Font(bold=True, size=13, color="1F4E79")
bold_font      = Font(bold=True, size=10)
normal_font    = Font(size=10)

thin_border = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

def style_header_row(ws, row_num, col_count):
    for col in range(1, col_count + 1):
        cell = ws.cell(row=row_num, column=col)
        cell.fill   = HEADER
        cell.font   = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border

def apply_border(ws, row_num, col_count):
    for col in range(1, col_count + 1):
        ws.cell(row=row_num, column=col).border = thin_border

# ══════════════════════════════════════════════════════════════════════════════
# TAB 1 — SUMMARY DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = "Summary Dashboard"
ws1.column_dimensions["A"].width = 30
ws1.column_dimensions["B"].width = 20

# Title
ws1.merge_cells("A1:B1")
ws1["A1"] = "Catalogue Quality Audit Report"
ws1["A1"].font   = Font(bold=True, size=15, color="1F4E79")
ws1["A1"].alignment = Alignment(horizontal="center")
ws1.row_dimensions[1].height = 30

# Section: Overall Metrics
ws1["A3"] = "OVERALL METRICS"
ws1["A3"].font = Font(bold=True, size=11, color="1F4E79")

metrics = [
    ("Total SKUs Audited",        total_skus),
    ("Unique SKUs",                unique_skus),
    ("Duplicate SKUs Found",       duplicates),
    ("Avg Completeness Score (%)", avg_score),
    ("Good Quality Listings",      good),
    ("Needs Review",               needs_review),
    ("Poor Quality Listings",      poor),
]

for i, (label, value) in enumerate(metrics, start=4):
    ws1[f"A{i}"] = label
    ws1[f"B{i}"] = value
    ws1[f"A{i}"].font   = bold_font
    ws1[f"B{i}"].font   = normal_font
    ws1[f"B{i}"].alignment = Alignment(horizontal="center")
    fill = GREY if i % 2 == 0 else PatternFill("solid", fgColor="FFFFFF")
    ws1[f"A{i}"].fill = fill
    ws1[f"B{i}"].fill = fill
    apply_border(ws1, i, 2)

# Section: Missing Fields Breakdown
ws1["A12"] = "MISSING VALUES PER FIELD"
ws1["A12"].font = Font(bold=True, size=11, color="1F4E79")

ws1["A13"] = "Field"
ws1["B13"] = "Missing Count"
style_header_row(ws1, 13, 2)

for i, (field, count) in enumerate(missing_per_field.items(), start=14):
    ws1[f"A{i}"] = field
    ws1[f"B{i}"] = count
    ws1[f"A{i}"].font = normal_font
    ws1[f"B{i}"].font = normal_font
    ws1[f"B{i}"].alignment = Alignment(horizontal="center")
    ws1[f"B{i}"].fill = RED if count > 0 else GREEN
    apply_border(ws1, i, 2)

# ══════════════════════════════════════════════════════════════════════════════
# TAB 2 — FULL AUDIT DETAIL
# ══════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Full Audit Detail")

cols = list(df.columns)
for col_num, col_name in enumerate(cols, start=1):
    ws2.cell(row=1, column=col_num, value=col_name)
style_header_row(ws2, 1, len(cols))

flag_col = cols.index("Quality_Flag") + 1
score_col = cols.index("Completeness_Score") + 1

for row_num, row in enumerate(df.itertuples(index=False), start=2):
    for col_num, value in enumerate(row, start=1):
        cell = ws2.cell(row=row_num, column=col_num, value=value)
        cell.font   = normal_font
        cell.border = thin_border
        cell.alignment = Alignment(horizontal="left", vertical="center")

    flag = ws2.cell(row=row_num, column=flag_col).value
    if flag == "Good":
        ws2.cell(row=row_num, column=flag_col).fill = GREEN
    elif flag == "Needs Review":
        ws2.cell(row=row_num, column=flag_col).fill = YELLOW
    elif flag in ("Poor", "Duplicate"):
        ws2.cell(row=row_num, column=flag_col).fill = RED

    score = ws2.cell(row=row_num, column=score_col).value
    if score == 100:
        ws2.cell(row=row_num, column=score_col).fill = GREEN
    elif score >= 70:
        ws2.cell(row=row_num, column=score_col).fill = YELLOW
    else:
        ws2.cell(row=row_num, column=score_col).fill = RED

for col_num in range(1, len(cols) + 1):
    ws2.column_dimensions[get_column_letter(col_num)].width = 22

ws2.freeze_panes = "A2"

# ══════════════════════════════════════════════════════════════════════════════
# TAB 3 — POOR & DUPLICATE SKUS ONLY
# ══════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet("Action Required")

flagged_df = df[df["Quality_Flag"].isin(["Poor", "Duplicate"])].reset_index(drop=True)

for col_num, col_name in enumerate(cols, start=1):
    ws3.cell(row=1, column=col_num, value=col_name)
style_header_row(ws3, 1, len(cols))

for row_num, row in enumerate(flagged_df.itertuples(index=False), start=2):
    for col_num, value in enumerate(row, start=1):
        cell = ws3.cell(row=row_num, column=col_num, value=value)
        cell.font   = normal_font
        cell.border = thin_border
        cell.alignment = Alignment(horizontal="left")
    ws3.cell(row=row_num, column=flag_col).fill = RED

for col_num in range(1, len(cols) + 1):
    ws3.column_dimensions[get_column_letter(col_num)].width = 22

ws3.freeze_panes = "A2"

# ── 8. SAVE ───────────────────────────────────────────────────────────────────
output_path = "catalogue_audit_report.xlsx"
wb.save(output_path)
print(f"Audit report saved to {output_path}")
print(f"\nSummary:")
print(f"  Total SKUs      : {total_skus}")
print(f"  Duplicates      : {duplicates}")
print(f"  Good            : {good}")
print(f"  Needs Review    : {needs_review}")
print(f"  Poor            : {poor}")
print(f"  Avg Score       : {avg_score}%")

Audit report saved to catalogue_audit_report.xlsx

Summary:
  Total SKUs      : 108
  Duplicates      : 8
  Good            : 11
  Needs Review    : 47
  Poor            : 42
  Avg Score       : 83.4%
